In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("Sales Summary validation").getOrCreate()


c:\Users\LENOVO\Desktop\AdvancedPySparkPipeline\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
gold_df = spark.read.parquet("../../data/gold/sales_summary")

In [4]:
gold_df.printSchema()

root
 |-- order_date: date (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- total_items_sold: long (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- total_product_sales: double (nullable = true)
 |-- total_freight: double (nullable = true)
 |-- avg_order_value: double (nullable = true)
 |-- previous_day_revenue: double (nullable = true)
 |-- revenue_growth: double (nullable = true)
 |-- cumulative_revenue: double (nullable = true)
 |-- 7_day_avg_revenue: double (nullable = true)



In [5]:
print("Gold rows:", gold_df.count())

Gold rows: 615


In [6]:
from pyspark.sql.functions import col, sum
gold_df.select(
    sum(col("order_date").isNull().cast("int")).alias("null_dates"),
    sum(col("total_revenue").isNull().cast("int")).alias("null_revenue")
).show()

+----------+------------+
|null_dates|null_revenue|
+----------+------------+
|         0|           0|
+----------+------------+



In [7]:
gold_df.filter(
    col("total_revenue") < 0
).show()

+----------+------------+----------------+-------------+-------------------+-------------+---------------+--------------------+--------------+------------------+-----------------+
|order_date|total_orders|total_items_sold|total_revenue|total_product_sales|total_freight|avg_order_value|previous_day_revenue|revenue_growth|cumulative_revenue|7_day_avg_revenue|
+----------+------------+----------------+-------------+-------------------+-------------+---------------+--------------------+--------------+------------------+-----------------+
+----------+------------+----------------+-------------+-------------------+-------------+---------------+--------------------+--------------+------------------+-----------------+



In [8]:
gold_df.filter(
    col("total_orders") <=0
).show()

+----------+------------+----------------+-------------+-------------------+-------------+---------------+--------------------+--------------+------------------+-----------------+
|order_date|total_orders|total_items_sold|total_revenue|total_product_sales|total_freight|avg_order_value|previous_day_revenue|revenue_growth|cumulative_revenue|7_day_avg_revenue|
+----------+------------+----------------+-------------+-------------------+-------------+---------------+--------------------+--------------+------------------+-----------------+
+----------+------------+----------------+-------------+-------------------+-------------+---------------+--------------------+--------------+------------------+-----------------+



In [9]:
gold_df.filter(
    col("total_items_sold") < col("total_orders")
).show()

+----------+------------+----------------+-------------+-------------------+-------------+---------------+--------------------+--------------+------------------+-----------------+
|order_date|total_orders|total_items_sold|total_revenue|total_product_sales|total_freight|avg_order_value|previous_day_revenue|revenue_growth|cumulative_revenue|7_day_avg_revenue|
+----------+------------+----------------+-------------+-------------------+-------------+---------------+--------------------+--------------+------------------+-----------------+
+----------+------------+----------------+-------------+-------------------+-------------+---------------+--------------------+--------------+------------------+-----------------+



In [10]:
orders = spark.read.parquet("../../data/silver/orders")
order_items = spark.read.parquet("../../data/silver/order_items")
payments = spark.read.parquet("../../data/silver/payments")

In [11]:
payments.groupBy("order_id").count().orderBy(col("count").desc()).show(10)

+--------------------+-----+
|            order_id|count|
+--------------------+-----+
|fa65dad1b0e818e3c...|   29|
|ccf804e764ed5650c...|   26|
|285c2e15bebd4ac83...|   22|
|895ab968e7bb0d565...|   21|
|fedcd9f7ccdc8cba3...|   19|
|ee9ca989fc93ba09a...|   19|
|21577126c19bf11a0...|   15|
|4bfcba9e084f46c8e...|   15|
|3c58bffb70dcf45f1...|   14|
|4689b1816de42507a...|   14|
+--------------------+-----+
only showing top 10 rows


In [12]:
order_items.groupBy("order_id") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(10)

+--------------------+-----+
|            order_id|count|
+--------------------+-----+
|8272b63d03f5f79c5...|   21|
|1b15974a0141d54e3...|   20|
|ab14fdcfbe524636d...|   20|
|428a2f660dc84138d...|   15|
|9ef13efd6949e4573...|   15|
|73c8ab38f07dc9438...|   14|
|9bdc4d4c71aa1de46...|   14|
|37ee401157a3a0b28...|   13|
|2c2a19b5703863c90...|   12|
|637617b3ffe9e2f7a...|   12|
+--------------------+-----+
only showing top 10 rows


In [13]:
gold_df.show()

+----------+------------+----------------+-------------+-------------------+-------------+---------------+--------------------+--------------+------------------+-----------------+
|order_date|total_orders|total_items_sold|total_revenue|total_product_sales|total_freight|avg_order_value|previous_day_revenue|revenue_growth|cumulative_revenue|7_day_avg_revenue|
+----------+------------+----------------+-------------+-------------------+-------------+---------------+--------------------+--------------+------------------+-----------------+
|2016-09-04|           1|               2|       136.23|              72.89|        63.34|         136.23|                NULL|          NULL|            136.23|           136.23|
|2016-09-05|           1|               1|        75.06|               59.5|        15.56|          75.06|              136.23|        -61.17|            211.29|           105.65|
|2016-10-02|           1|               1|       109.34|              100.0|         9.34|         1

In [14]:
gold_df.printSchema()

root
 |-- order_date: date (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- total_items_sold: long (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- total_product_sales: double (nullable = true)
 |-- total_freight: double (nullable = true)
 |-- avg_order_value: double (nullable = true)
 |-- previous_day_revenue: double (nullable = true)
 |-- revenue_growth: double (nullable = true)
 |-- cumulative_revenue: double (nullable = true)
 |-- 7_day_avg_revenue: double (nullable = true)

